# NIFTY Options IV Surface Reconstruction — Winning Submission

**Method:** cross-sectional smile fit **+** averaged low-variance residual ensemble.

```
final_IV = cs_predict(smile)  +  mean( HGBM_bag , RandomForest , ExtraTrees )
```
Each of the three learners is trained to predict the residual `actual − cs_predict` over all observed cells; their predictions are averaged and added back to the smile fit.

**Why this design** (each piece earned its place on the public leaderboard):
- `cs_predict` — local volatility-smile interpolation does the heavy lifting; it's the signal that transfers from liquid (observed) to illiquid (missing) strikes.
- **residual ensemble** — a single global HGBM correction helped; CatBoost (deeper, less regularized) overfit and hurt. So the correction uses three *low-variance* learners — bagged HGBM, random forest, extra-trees — **averaged**. Diversity + bagging = variance reduction, which is what generalizes to the illiquid test cells. This averaged ensemble beat every single-model submission.

**Guardrails:** strictly-causal features (no feature reads a future value of the target cell; global *training* across timestamps is fine for imputation). Don't tune on `sandbox_solution.csv` (not the answer key). No time-interpolation (it reads the future = look-ahead bias). `self_validate()` only masks liquid cells, so it over-rates complex models — trust the public board for selection.

Reproducible (`random_state=0`).

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import (HistGradientBoostingRegressor, BaggingRegressor,
                              RandomForestRegressor, ExtraTreesRegressor)

df = pd.read_csv('dataset.csv')
iv_cols = [c for c in df.columns if c not in ('datetime', 'underlying_price')]
ce_cols = [c for c in iv_cols if c.endswith('CE')]
pe_cols = [c for c in iv_cols if c.endswith('PE')]
strike  = lambda c: int(c[12:-2])
print(f'Loaded: {len(df)} rows, {len(iv_cols)} IV cols ({len(ce_cols)} CE, {len(pe_cols)} PE)')
print(f'Missing cells to fill: {df[iv_cols].isna().sum().sum()}')

In [ ]:
def cs_predict(row, col, group, is_ce):
    """IV at strike k0 from the same-timestamp smile (inverse-distance weighted quadratic
    over the 4 nearest strikes per side, with a linear-interpolation sanity fallback)."""
    k0 = strike(col)
    obs = sorted([(strike(c), float(row[c])) for c in group
                  if c != col and pd.notna(row[c]) and float(row[c]) > 0], key=lambda x: x[0])
    if len(obs) < 2:
        return np.nan
    bl = sorted([(k, v) for k, v in obs if k < k0], key=lambda x: -x[0])
    ab = sorted([(k, v) for k, v in obs if k > k0], key=lambda x:  x[0])
    if bl and ab:
        pts = bl[:4] + ab[:4]; pts.sort(key=lambda x: x[0])
        sk = np.array([p[0] for p in pts]); sv = np.array([p[1] for p in pts])
        try:
            d = np.abs(sk - k0).astype(float); d[d < 50] = 50
            cf = np.polyfit(sk, sv, min(2, len(sk) - 1), w=1.0 / d)
            pred = float(np.polyval(cf, k0))
            lmin = min(sv[np.argmin(np.abs(sk - k0))], sv.min())
            lmax = max(sv[np.argmax(np.abs(sk - k0))], sv.max())
            if pred < lmin * 0.5 or pred > lmax * 2.0:
                lK, lIV = bl[0]; rK, rIV = ab[0]
                return lIV + (k0 - lK) / (rK - lK) * (rIV - lIV)
            return pred
        except Exception:
            lK, lIV = bl[0]; rK, rIV = ab[0]
            return lIV + (k0 - lK) / (rK - lK) * (rIV - lIV)
    side_all = sorted(bl if bl else ab, key=lambda x: abs(x[0] - k0))
    obs_ks = [s[0] for s in side_all]
    going_otm = (k0 > max(obs_ks)) if is_ce else (k0 < min(obs_ks))
    side = side_all[:3] if going_otm else side_all[:2]; side.sort(key=lambda x: x[0])
    sk = [p[0] for p in side]; sv = [p[1] for p in side]
    try:
        return float(np.polyval(np.polyfit(sk, sv, min(1, len(sk) - 1)), k0))
    except Exception:
        return side[0][1]

In [ ]:
def build_features(data):
    """Per-cell feature matrix (13 features, all same-timestamp or strictly-past)."""
    N = len(data); und = data['underlying_price'].values
    feats, y, keys, cs = [], [], [], []
    for i in range(N):
        row = data.iloc[i]; F = und[i]
        obs_all = [row[c] for c in iv_cols if pd.notna(row[c]) and row[c] > 0]
        atm   = float(np.median(obs_all)) if obs_all else 0.15
        ivstd = float(np.std(obs_all))    if obs_all else 0.0
        for col in iv_cols:
            ic = col.endswith('CE'); g = ce_cols if ic else pe_cols; k0 = strike(col)
            p = cs_predict(row, col, g, ic); pp = p if pd.notna(p) else atm
            obs = sorted([(strike(c), float(row[c])) for c in g
                          if c != col and pd.notna(row[c]) and row[c] > 0])
            below = [o for o in obs if o[0] < k0]; above = [o for o in obs if o[0] > k0]
            nb = below[-1] if below else (np.nan, np.nan)
            na = above[0] if above else (np.nan, np.nan)
            past = data[col].iloc[:i]; li = past.last_valid_index()
            cf = float(data[col].iloc[li]) if li is not None else atm
            feats.append([
                np.log(k0 / F), k0, 1.0 if ic else 0.0, F, i / N, atm, ivstd,
                pp, cf,
                nb[1] if not np.isnan(nb[1]) else atm, (k0 - nb[0]) if not np.isnan(nb[0]) else 999,
                na[1] if not np.isnan(na[1]) else atm, (na[0] - k0) if not np.isnan(na[0]) else 999,
            ])
            keys.append((i, col)); cs.append(pp)
            y.append(float(row[col]) if pd.notna(row[col]) else np.nan)
    return np.array(feats), keys, np.array(y), np.array(cs)

def _models():
    hgb = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05,
            l2_regularization=1.0, min_samples_leaf=20, random_state=0)
    return [
        BaggingRegressor(estimator=hgb, n_estimators=15, max_samples=0.75,
                         max_features=0.75, random_state=0, n_jobs=-1),
        RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                              max_features=0.5, random_state=0, n_jobs=-1),
        ExtraTreesRegressor(n_estimators=300, min_samples_leaf=15,
                            max_features=0.5, random_state=0, n_jobs=-1),
    ]

def fit_predict(data, target_mask):
    """Train the 3-model residual ensemble on observed cells; predict target_mask cells."""
    X, keys, y, cs = build_features(data)
    observed  = ~np.isnan(y)
    is_target = np.array([target_mask.loc[i, c] for (i, c) in keys])
    train = observed & (~is_target)
    Xtr, ytr = X[train], (y - cs)[train]
    resid = np.zeros(len(keys))
    for m in _models():
        m.fit(Xtr, ytr)
        resid += m.predict(X)
    resid /= 3.0
    return {(i, c): max(0.005, float(cs[idx] + resid[idx]))
            for idx, (i, c) in enumerate(keys) if target_mask.loc[i, c]}

In [ ]:
def self_validate(frac=0.15, seed=0):
    """Mask known cells, predict, score vs truth (liquid-cell sanity check only)."""
    rng = np.random.default_rng(seed)
    holdout = pd.DataFrame(False, index=df.index, columns=iv_cols)
    for col in iv_cols:
        obs_idx = df.index[df[col].notna()].to_numpy()
        holdout.loc[rng.choice(obs_idx, size=max(1, int(frac*len(obs_idx))), replace=False), col] = True
    work = df.copy(); work[iv_cols] = df[iv_cols].mask(holdout)
    preds = fit_predict(work, holdout)
    mse = float(np.mean([(preds[(i, c)] - df.loc[i, c]) ** 2 for (i, c) in preds]))
    print(f'Self-validation MSE ({len(preds)} held-out liquid cells, seed {seed}): {mse:.6f}')
    return mse

self_validate(seed=0)

In [ ]:
# Fit on all observed cells, predict the missing ones
missing_mask = df[iv_cols].isna()
preds = fit_predict(df, missing_mask)
filled = df.copy()
for (i, c), v in preds.items():
    filled.at[i, c] = v
print(f'Filled {len(preds)} cells; remaining NaN: {filled[iv_cols].isna().sum().sum()}')
filled.to_csv('filled_dataset.csv', index=False)

def generate_solution(filled_path, output_path='submission.csv'):
    original  = pd.read_csv('dataset.csv')
    filled_df = pd.read_csv(filled_path)
    rows = []
    for col in [c for c in original.columns if c != 'datetime']:
        for idx in original.index[original[col].isna()]:
            dt = original.loc[idx, 'datetime']
            rows.append({'id': f'{dt}||{col}', 'value': filled_df.loc[idx, col]})
    sol = pd.DataFrame(rows, columns=['id', 'value']).sort_values('id').reset_index(drop=True)
    sol.to_csv(output_path, index=False)
    print(f'Solution saved -> {output_path}  ({len(sol)} rows)')
    return sol

sub = generate_solution('filled_dataset.csv', 'submission.csv')
sub.head()